In [1]:
import numpy as np
from scipy.stats import binom
import plotly.graph_objects as go

In [2]:
# define biases
A_BIAS = 0.65 # unknown (true bias of A)
B_BIAS = 0.55 # unknown (true bias of B)

# number of coinflips per trial
N = 20

# generate trials
NUM_A_TRIALS = 20 # unknown
NUM_B_TRIALS = 20 # unknown
trials = []

# num_trials = NUM_A_TRIALS + NUM_B_TRIALS
T = NUM_A_TRIALS + NUM_B_TRIALS

n_heads_A = np.random.binomial(N, A_BIAS, NUM_A_TRIALS)
n_heads_B = np.random.binomial(N, B_BIAS, NUM_B_TRIALS)

print('shape n_head_A:', n_heads_A.shape) # 20
print('shape n_head_B:', n_heads_B.shape) # 20

for i in range(NUM_A_TRIALS):
    # only n_heads value is known, not coin
    trials.append({"n_heads": n_heads_A[i], "coin": "A"})

for i in range(NUM_B_TRIALS):
    # only n_heads value is known, not coin
    trials.append({"n_heads": n_heads_B[i], "coin": "B"})

for i in trials:
    print(i)

shape n_head_A: (20,)
shape n_head_B: (20,)
{'n_heads': np.int32(12), 'coin': 'A'}
{'n_heads': np.int32(12), 'coin': 'A'}
{'n_heads': np.int32(11), 'coin': 'A'}
{'n_heads': np.int32(11), 'coin': 'A'}
{'n_heads': np.int32(11), 'coin': 'A'}
{'n_heads': np.int32(10), 'coin': 'A'}
{'n_heads': np.int32(13), 'coin': 'A'}
{'n_heads': np.int32(18), 'coin': 'A'}
{'n_heads': np.int32(15), 'coin': 'A'}
{'n_heads': np.int32(14), 'coin': 'A'}
{'n_heads': np.int32(16), 'coin': 'A'}
{'n_heads': np.int32(13), 'coin': 'A'}
{'n_heads': np.int32(14), 'coin': 'A'}
{'n_heads': np.int32(19), 'coin': 'A'}
{'n_heads': np.int32(11), 'coin': 'A'}
{'n_heads': np.int32(14), 'coin': 'A'}
{'n_heads': np.int32(12), 'coin': 'A'}
{'n_heads': np.int32(13), 'coin': 'A'}
{'n_heads': np.int32(14), 'coin': 'A'}
{'n_heads': np.int32(16), 'coin': 'A'}
{'n_heads': np.int32(14), 'coin': 'B'}
{'n_heads': np.int32(13), 'coin': 'B'}
{'n_heads': np.int32(14), 'coin': 'B'}
{'n_heads': np.int32(9), 'coin': 'B'}
{'n_heads': np.int32(

In [4]:
# initialize guesses

fig = go.Figure()

# guess bias towards heads
p_A = 0.5
p_B = 0.5

# guess of selecting each coin for each trial (init values)
pi_A_new = 0.5
pi_B_new = 0.5

# guesses for which coins generated which trial
coin_guesses = []

N_ITERATIONS = 5

p_A_estimates = [p_A]
p_B_estimates = [p_B]

pi_A_estimates = [pi_A_new]
pi_B_estimates = [pi_B_new]

ll_estimates = []

for i in range(N_ITERATIONS):
    print('iteration:', i)

    p_A_new_N = 0
    p_A_new_D = 0
    p_B_new_N = 0
    p_B_new_D = 0

    term_for_ll = 0.0

    # iterate over trials, and compute (normalized) probabilities for coin flips, as
    # well as the new values for p_A and p_B
    for j, t in enumerate(trials):
        print('trial:'); print(t)

        # e-step
        prob_A = binom.pmf(t["n_heads"], N, p_A)
        prob_B = binom.pmf(t["n_heads"], N, p_B)

        prob_A_norm = prob_A / (prob_A + prob_B)
        prob_B_norm = prob_B / (prob_A + prob_B)

        print('prob A and B norm:', prob_A_norm, prob_B_norm)

        # m-step (cumulative)
        p_A_new_N += prob_A_norm * t["n_heads"]
        p_A_new_D += prob_A_norm

        p_B_new_N += prob_B_norm * t["n_heads"]
        p_B_new_D += prob_B_norm

        coin_guesses.append("A" if prob_A > prob_B else "B")

        term_for_ll += np.log(
            pi_A_new * binom.pmf(t["n_heads"], N, p_A) +
            pi_B_new * binom.pmf(t["n_heads"], N, p_B)
            )

    print()
    
    # m-step
    p_A = p_A_new_N / (p_A_new_D * N)
    p_B = p_B_new_N / (p_B_new_D * N)

    p_A_estimates.append(p_A)
    p_B_estimates.append(p_B)

    pi_A_new = p_A_new_D / T
    pi_B_new = p_B_new_D / T

    ll_estimates.append(term_for_ll)

    print('new estimated biases:', p_A, p_B)

print('coin guesses and correct values:')
c = 0
for k in range(NUM_A_TRIALS + NUM_A_TRIALS):
    print('guessed:', coin_guesses[k], ", correct:", trials[k]["coin"])
    if coin_guesses[k] == trials[k]["coin"]:
        c += 1

print('% correct coin guesses:')
print(c / (NUM_A_TRIALS + NUM_B_TRIALS) * 100, "%")

n = len(p_A_estimates)

fig.add_trace(go.Scatter(x=list(range(n)), y=p_A_estimates, name="A bias estimates"))
fig.add_trace(go.Scatter(x=list(range(n)), y=p_B_estimates, name="B estimate bias"))
fig.add_trace(go.Scatter(x=list(range(n)), y=[A_BIAS for _ in range(N_ITERATIONS + 1)], name="A real bias"))
fig.add_trace(go.Scatter(x=list(range(n)), y=[B_BIAS for _ in range(N_ITERATIONS + 1)], name="B real bias"))



fig.show()





iteration: 0
trial:
{'n_heads': np.int32(12), 'coin': 'A'}
prob A and B norm: 0.5 0.5
trial:
{'n_heads': np.int32(12), 'coin': 'A'}
prob A and B norm: 0.5 0.5
trial:
{'n_heads': np.int32(11), 'coin': 'A'}
prob A and B norm: 0.5 0.5
trial:
{'n_heads': np.int32(11), 'coin': 'A'}
prob A and B norm: 0.5 0.5
trial:
{'n_heads': np.int32(11), 'coin': 'A'}
prob A and B norm: 0.5 0.5
trial:
{'n_heads': np.int32(10), 'coin': 'A'}
prob A and B norm: 0.5 0.5
trial:
{'n_heads': np.int32(13), 'coin': 'A'}
prob A and B norm: 0.5 0.5
trial:
{'n_heads': np.int32(18), 'coin': 'A'}
prob A and B norm: 0.5 0.5
trial:
{'n_heads': np.int32(15), 'coin': 'A'}
prob A and B norm: 0.5 0.5
trial:
{'n_heads': np.int32(14), 'coin': 'A'}
prob A and B norm: 0.5 0.5
trial:
{'n_heads': np.int32(16), 'coin': 'A'}
prob A and B norm: 0.5 0.5
trial:
{'n_heads': np.int32(13), 'coin': 'A'}
prob A and B norm: 0.5 0.5
trial:
{'n_heads': np.int32(14), 'coin': 'A'}
prob A and B norm: 0.5 0.5
trial:
{'n_heads': np.int32(19), 'coin

In [5]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(n)), y=ll_estimates, name="Log likelyhoods"))
fig.show()